**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (Install)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

In [ ]:
data.head()

,Package_Name,Total_Dependency_Count,Total_Dependencies,Direct_Dependency_Count,Direct_Dependencies,Indirect_Dependency_Count,Indirect_Dependencies,Level
0,10Cent10-999.0.4.tar.gz,0,NaN,0,NaN,0,NaN,1
1,10Cent11-999.0.4.tar.gz,0,NaN,0,NaN,0,NaN,1
2,11Cent-999.0.0.tar.gz,0,NaN,0,NaN,0,NaN,1
3,11Cent-999.0.1.tar.gz,0,NaN,0,NaN,0,NaN,1
4,11Cent-999.0.2.tar.gz,0,NaN,0,NaN,0,NaN,1


In [ ]:
if 'Package_Name' in data.columns:
    data = data.drop(columns=['Package_Name'])

# Combine categorical columns into a single text feature
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col != 'Level']
data['Combined_Categorical'] = data[categorical_columns].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the combined categorical text column using n-grams
vectorizer = CountVectorizer(ngram_range=(2, 3))
categorical_ngrams = vectorizer.fit_transform(data['Combined_Categorical'])

# Ensure only valid numeric columns are selected
numerical_columns = [
    col for col in data.columns
    if pd.to_numeric(data[col], errors='coerce').notnull().all() and col != 'Level'
]

# Prepare numerical features
numerical_features = data[numerical_columns].fillna(0).astype(float)

# Convert numerical features to sparse matrix
numerical_features_sparse = csr_matrix(numerical_features.values)

# Combine numerical features with n-gram features
X = hstack([numerical_features_sparse, categorical_ngrams])

# Scale features before combining
scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
X_scaled = scaler.fit_transform(X)

In [ ]:

# Assume 'data' is your pandas DataFrame
X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 08-29 20:46:24] {1680} INFO - task = classification
[flaml.automl.logger: 08-29 20:46:24] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 08-29 20:46:24] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 08-29 20:46:24] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 08-29 20:46:24] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-29 20:46:24] {2346} INFO - Estimated sufficient time budget=1930s. Estimated necessary time budget=44s.
[flaml.automl.logger: 08-29 20:46:24] {2398} INFO -  at 0.2s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:24] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-29 20:46:24] {2398} INFO -  at 0.4s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:

[flaml.automl.logger: 08-29 20:46:38] {2219} INFO - iteration 34, current learner lgbm
[flaml.automl.logger: 08-29 20:46:38] {2398} INFO -  at 14.3s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:38] {2219} INFO - iteration 35, current learner rf
[flaml.automl.logger: 08-29 20:46:39] {2398} INFO -  at 15.6s,	estimator rf's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:39] {2219} INFO - iteration 36, current learner extra_tree
[flaml.automl.logger: 08-29 20:46:41] {2398} INFO -  at 17.4s,	estimator extra_tree's best error=0.2625,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:41] {2219} INFO - iteration 37, current learner xgboost
[flaml.automl.logger: 08-29 20:46:41] {2398} INFO -  at 17.5s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:41] {2219} INFO - iteration 38, current learner 

[flaml.automl.logger: 08-29 20:46:54] {2219} INFO - iteration 70, current learner xgboost
[flaml.automl.logger: 08-29 20:46:54] {2398} INFO -  at 30.5s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:54] {2219} INFO - iteration 71, current learner xgboost
[flaml.automl.logger: 08-29 20:46:54] {2398} INFO -  at 30.6s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:54] {2219} INFO - iteration 72, current learner extra_tree
[flaml.automl.logger: 08-29 20:46:56] {2398} INFO -  at 31.9s,	estimator extra_tree's best error=0.2539,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:56] {2219} INFO - iteration 73, current learner xgboost
[flaml.automl.logger: 08-29 20:46:56] {2398} INFO -  at 32.0s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:46:56] {2219} INFO - iteration 74, 

[flaml.automl.logger: 08-29 20:47:15] {2219} INFO - iteration 106, current learner xgboost
[flaml.automl.logger: 08-29 20:47:15] {2398} INFO -  at 51.3s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:15] {2219} INFO - iteration 107, current learner rf
[flaml.automl.logger: 08-29 20:47:16] {2398} INFO -  at 52.6s,	estimator rf's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:16] {2219} INFO - iteration 108, current learner xgboost
[flaml.automl.logger: 08-29 20:47:17] {2398} INFO -  at 52.7s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:17] {2219} INFO - iteration 109, current learner xgboost
[flaml.automl.logger: 08-29 20:47:17] {2398} INFO -  at 52.8s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:17] {2219} INFO - iteration 110, current lea

[flaml.automl.logger: 08-29 20:47:30] {2219} INFO - iteration 142, current learner xgboost
[flaml.automl.logger: 08-29 20:47:30] {2398} INFO -  at 65.9s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:30] {2219} INFO - iteration 143, current learner xgboost
[flaml.automl.logger: 08-29 20:47:30] {2398} INFO -  at 66.0s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:30] {2219} INFO - iteration 144, current learner xgboost
[flaml.automl.logger: 08-29 20:47:30] {2398} INFO -  at 66.2s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:30] {2219} INFO - iteration 145, current learner lgbm
[flaml.automl.logger: 08-29 20:47:30] {2398} INFO -  at 66.4s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:47:30] {2219} INFO - iteration 146, current

...

[flaml.automl.logger: 08-29 20:49:44] {2219} INFO - iteration 527, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:49:44] {2398} INFO -  at 200.6s,	estimator xgb_limitdepth's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:44] {2219} INFO - iteration 528, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:49:44] {2398} INFO -  at 200.7s,	estimator xgb_limitdepth's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:44] {2219} INFO - iteration 529, current learner xgboost
[flaml.automl.logger: 08-29 20:49:45] {2398} INFO -  at 200.8s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:45] {2219} INFO - iteration 530, current learner lgbm
[flaml.automl.logger: 08-29 20:49:45] {2398} INFO -  at 201.0s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:45] {221

[flaml.automl.logger: 08-29 20:49:52] {2219} INFO - iteration 562, current learner extra_tree
[flaml.automl.logger: 08-29 20:49:53] {2398} INFO -  at 209.6s,	estimator extra_tree's best error=0.2498,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:53] {2219} INFO - iteration 563, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:49:54] {2398} INFO -  at 209.9s,	estimator xgb_limitdepth's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:54] {2219} INFO - iteration 564, current learner lgbm
[flaml.automl.logger: 08-29 20:49:54] {2398} INFO -  at 210.1s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:54] {2219} INFO - iteration 565, current learner lgbm
[flaml.automl.logger: 08-29 20:49:54] {2398} INFO -  at 210.3s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:49:54] {2219} INFO - iter

[flaml.automl.logger: 08-29 20:50:23] {2398} INFO -  at 239.2s,	estimator extra_tree's best error=0.2495,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:23] {2219} INFO - iteration 597, current learner extra_tree
[flaml.automl.logger: 08-29 20:50:26] {2398} INFO -  at 241.7s,	estimator extra_tree's best error=0.2495,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:26] {2219} INFO - iteration 598, current learner rf
[flaml.automl.logger: 08-29 20:50:28] {2398} INFO -  at 244.2s,	estimator rf's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:28] {2219} INFO - iteration 599, current learner rf
[flaml.automl.logger: 08-29 20:50:31] {2398} INFO -  at 246.8s,	estimator rf's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:31] {2219} INFO - iteration 600, current learner rf
[flaml.automl.logger: 08-29 20:50:33] {2398} INFO -  at 249.4s,	estimator rf's b

[flaml.automl.logger: 08-29 20:50:57] {2398} INFO -  at 273.0s,	estimator extra_tree's best error=0.2495,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:57] {2219} INFO - iteration 632, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:50:57] {2398} INFO -  at 273.2s,	estimator xgb_limitdepth's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:57] {2219} INFO - iteration 633, current learner extra_tree
[flaml.automl.logger: 08-29 20:50:59] {2398} INFO -  at 275.7s,	estimator extra_tree's best error=0.2495,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:50:59] {2219} INFO - iteration 634, current learner extra_tree
[flaml.automl.logger: 08-29 20:51:02] {2398} INFO -  at 278.2s,	estimator extra_tree's best error=0.2495,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:51:02] {2219} INFO - iteration 635, current learner xgboost
[flaml.automl.logger: 08-29 20:51:0

[flaml.automl.logger: 08-29 20:51:10] {2398} INFO -  at 286.1s,	estimator xgb_limitdepth's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:51:10] {2219} INFO - iteration 667, current learner lgbm
[flaml.automl.logger: 08-29 20:51:10] {2398} INFO -  at 286.3s,	estimator lgbm's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:51:10] {2219} INFO - iteration 668, current learner xgboost
[flaml.automl.logger: 08-29 20:51:10] {2398} INFO -  at 286.4s,	estimator xgboost's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:51:10] {2219} INFO - iteration 669, current learner rf
[flaml.automl.logger: 08-29 20:51:12] {2398} INFO -  at 288.0s,	estimator rf's best error=0.2494,	best estimator lgbm's best error=0.2494
[flaml.automl.logger: 08-29 20:51:12] {2219} INFO - iteration 670, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:51:12] {2398} INFO -  at 288.1s,	es

In [ ]:
selected_features = ['Direct_Dependencies', 'Indirect_Dependencies']